# Logistic regression
**Machine Learning · Week 3 (Classification) · Day 1**

Lesson page: https://sara-aminpour.com/tutorials/ml/w3d1.html

A straight line scores each point, the **sigmoid** turns the score into a probability,
and **cross-entropy** measures how good those probabilities are. Trained with PyTorch.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-8, 8, 200)
plt.plot(z, sigmoid(z), color="#7b5cfa", lw=3)
plt.axhline(0.5, color="#8a85a3", ls="--", lw=1)
plt.axvline(0, color="#8a85a3", ls="--", lw=1)
plt.xlabel("score z = w·x + b")
plt.ylabel("probability of class 1")
plt.title("The sigmoid squashes any number into (0, 1)")
plt.show()
print("sigmoid(-4), sigmoid(0), sigmoid(4) =", sigmoid(np.array([-4, 0, 4])).round(3))

## Data and a PyTorch model
`nn.Linear(2, 1)` computes the score $z = w_1x_1 + w_2x_2 + b$.
`BCEWithLogitsLoss` applies the sigmoid and the cross-entropy loss together (it's more accurate that way).

In [ ]:
import torch
from sklearn.datasets import make_blobs

torch.set_num_threads(1)
X, y = make_blobs(n_samples=200, centers=[(-1.2, -0.6), (1.0, 0.9)], cluster_std=0.9, random_state=4)
Xt = torch.tensor(X, dtype=torch.float32)
yt = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)

torch.manual_seed(0)
model = torch.nn.Linear(2, 1)
loss_fn = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

for step in range(201):
    optimizer.zero_grad()
    loss = loss_fn(model(Xt), yt)
    loss.backward()
    optimizer.step()
    if step % 50 == 0:
        print(f"step {step:3d}: cross-entropy = {loss.item():.3f}")

with torch.no_grad():
    accuracy = ((model(Xt) > 0).float() == yt).float().mean().item()
print(f"training accuracy = {accuracy:.3f}")

## The probability map
Every point of the plane gets a probability. The decision boundary ($p = 0.5$) is a straight line.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

apricot_blue = LinearSegmentedColormap.from_list("apricot_blue", ["#ff8a50", "#ffffff", "#5577f5"])
xx, yy = np.meshgrid(np.linspace(-4.5, 4.5, 200), np.linspace(-4, 4.5, 200))
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
with torch.no_grad():
    p = torch.sigmoid(model(grid)).reshape(xx.shape).numpy()

fig, ax = plt.subplots(layout="constrained")
filled = ax.contourf(xx, yy, p, levels=20, cmap=apricot_blue, alpha=0.6)
ax.contour(xx, yy, p, levels=[0.5], colors="#2c2842", linewidths=2)
ax.scatter(X[:, 0], X[:, 1], c=np.where(y == 0, "#ff8a50", "#5577f5"), s=22)
fig.colorbar(filled, ax=ax, label="P(class 1)")
ax.set_title("Logistic regression: probabilities and the boundary")
plt.show()

## The scikit-learn shortcut
Same model, one line. (scikit-learn also adds a little ridge penalty by default, the leash from last week.)

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression().fit(X, y)
print("weights:", clf.coef_.round(3), " bias:", clf.intercept_.round(3))
print("accuracy:", clf.score(X, y))
print("P(class 1) at the point (0, 0):", clf.predict_proba([[0.0, 0.0]])[0, 1].round(3))

## Your turn: a real medical dataset
The breast-cancer dataset describes 569 tumours with 30 measurements each; the label says whether the
tumour is malignant (0) or benign (1). The split and the standardizing are done. Build a logistic regression
with 30 inputs in PyTorch, train it, and report the **test** accuracy.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
Xtr, Xte, ytr, yte = train_test_split(data.data, data.target, test_size=0.25, random_state=0, stratify=data.target)
mu, sd = Xtr.mean(axis=0), Xtr.std(axis=0)                 # fitted on the training set only (no leakage!)
Xtr_t = torch.tensor((Xtr - mu) / sd, dtype=torch.float32)
Xte_t = torch.tensor((Xte - mu) / sd, dtype=torch.float32)
ytr_t = torch.tensor(ytr, dtype=torch.float32).reshape(-1, 1)
yte_t = torch.tensor(yte, dtype=torch.float32).reshape(-1, 1)

torch.manual_seed(0)
model = None    # TODO: a linear layer with 30 inputs and 1 output
# TODO: a BCEWithLogitsLoss, an SGD optimizer (lr = 0.1), and 300 training steps on (Xtr_t, ytr_t)

if model is not None:
    with torch.no_grad():
        print(f"test accuracy = {((model(Xte_t) > 0).float() == yte_t).float().mean().item():.3f}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
torch.manual_seed(0)
model = torch.nn.Linear(30, 1)
loss_fn = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
for step in range(300):
    optimizer.zero_grad()
    loss = loss_fn(model(Xtr_t), ytr_t)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    test_acc = ((model(Xte_t) > 0).float() == yte_t).float().mean().item()
print(f"PyTorch test accuracy      = {test_acc:.3f}")
print(f"scikit-learn test accuracy = {LogisticRegression(max_iter=1000).fit((Xtr - mu) / sd, ytr).score((Xte - mu) / sd, yte):.3f}")
w = model.weight.detach().numpy().ravel()
top = np.argsort(-np.abs(w))[:3]
print("most influential measurements:", ", ".join(f"{data.feature_names[i]} ({w[i]:+.2f})" for i in top))